# Fit a small circuit end to end

This tutorial generates synthetic fluorescence from a three-neuron circuit, perturbs one chemical strength, and fits that parameter with the existing Rust reference optimizer. A different stimulation pulse checks the fitted model afterward. No biological data, population fitting, parameter identifiability, or whole-worm accuracy is established here.

The graph, initial parameters, and teacher are deterministic; no random seed is used. All parameters except one chemical strength are fixed at their teacher values. This deliberately easy recovery exercise explains the fitting contract. Real-data fitting uses the [JAX backend](../backends/jax/README.md).

**Requirements:** Git, a current Rust toolchain, and a Python notebook kernel with Matplotlib. Start in the repository root or `tutorials/`. See [execution instructions](README.md).

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import subprocess
import sys
import matplotlib
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / "Cargo.toml").exists():
    root = root.parent
assert (root / "examples/fit_small.rs").is_file(), "Run from worm-sim or tutorials/"
run = root / "runs/tutorial-small"
run.mkdir(parents=True, exist_ok=True)
artifact = run / "fit.json"

def command(*args):
    return subprocess.check_output(args, cwd=root, text=True).strip()

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_paths = ["Cargo.toml", "Cargo.lock", "examples/fit_small.rs", "tutorials/fit-small-circuit.ipynb"]
source_paths += sorted(str(p.relative_to(root)) for p in (root / "src").rglob("*.rs"))
provenance = {
    "commit": command("git", "rev-parse", "HEAD"),
    "worktree_status": command("git", "status", "--porcelain"),
    "rustc": command("rustc", "--version"),
    "python": sys.version,
    "matplotlib": matplotlib.__version__,
    "source_sha256": {p: sha(root / p) for p in source_paths},
}
print("Source:", provenance["commit"])
print("Dirty worktree:", bool(provenance["worktree_status"]))

## Generate, perturb, and fit

`fit_small` constructs a ring of three synthetic neurons with one outgoing chemical edge per neuron and no gap junctions. It generates the teacher response to a pulse on N000 (0.1–0.3 s, amplitude 1), then adds 1.5 to the first unconstrained chemical-strength parameter. Positive strength is obtained through the model's softplus transform, so this is not an additive change in physical conductance.

The fit uses 80 Adam updates at learning rate 0.08, differentiating the fluorescence mean squared error with forward-mode AD. RK4 uses a 0.001 s step and saves every 0.01 s for 0.6 s. The artifact records the actual configuration, graph, parameters, and trajectories; the loss array includes the initial and final states.

In [ ]:
result = subprocess.run(
    ["cargo", "run", "--locked", "--release", "--example", "fit_small", "--", str(artifact)],
    cwd=root, text=True, capture_output=True, check=True,
)
print(result.stdout)
fit = json.loads(artifact.read_text())
provenance["artifact_sha256"] = sha(artifact)
(run / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
print(json.dumps(fit["optimizer"], indent=2))

## Check the contract before plotting

We recompute both errors from the exported arrays, verify that only the declared parameter changed, and require the training error to fall by at least tenfold. The separate pulse uses 0.2–0.4 s and amplitude 0.6. Its error is computed only after fitting; it is an engineering check with the same synthetic teacher, not an independent biological test set.

In [ ]:
def mse(a, b):
    assert len(a) == len(b) and len(a) > 0
    assert all(len(x) == len(y) and len(x) == 3 for x, y in zip(a, b))
    errors = [(x - y)**2 for row_a, row_b in zip(a, b) for x, y in zip(row_a, row_b)]
    return sum(errors) / len(errors)

history = fit["loss_history"]
train = fit["training"]
held = fit["held_pulse"]
assert len(history) == 81 and all(math.isfinite(v) and v >= 0 for v in history)
assert math.isclose(mse(train["truth"], train["initial"]), history[0], rel_tol=1e-10, abs_tol=1e-18)
assert math.isclose(mse(train["truth"], train["fitted"]), history[-1], rel_tol=1e-10, abs_tol=1e-18)
assert math.isclose(mse(held["truth"], held["fitted"]), held["mse"], rel_tol=1e-10, abs_tol=1e-18)
active = set(fit["optimizer"]["active_indices"])
p = fit["parameters"]
assert len(p["initial"]) == len(p["fitted"])
assert all(a == b for i, (a, b) in enumerate(zip(p["initial"], p["fitted"])) if i not in active)
assert history[-1] < history[0] / 10
print(f"Training MSE: {history[0]:.6e} → {history[-1]:.6e}")
print(f"Separate pulse MSE: {held['mse']:.6e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].semilogy(range(len(history)), history)
axes[0].set(xlabel="Completed Adam updates", ylabel="Fluorescence MSE", title="Training loss")
# Show the neuron whose training trace changed most from the perturbation.
neuron = max(range(3), key=lambda i: sum((a[i] - b[i])**2 for a, b in zip(train["truth"], train["initial"])))
for field, style in [("truth", "-"), ("initial", ":"), ("fitted", "--")]:
    axes[1].plot(train["times"], [row[neuron] for row in train[field]], style, label=field)
for field, style in [("truth", "-"), ("fitted", "--")]:
    axes[2].plot(held["times"], [row[neuron] for row in held[field]], style, label=field)
for ax, title in zip(axes[1:], ["Training pulse", "Separate pulse"]):
    ax.set(xlabel="Time (s)", ylabel="Synthetic fluorescence", title=f"{title}: N{neuron:03}")
    ax.legend()
fig.tight_layout()
fig.savefig(run / "fit.svg")
plt.show()

## What this demonstrates

The native graph → simulation → recording → selected-parameter fit → separate-pulse evaluation path runs end to end. `runs/tutorial-small/fit.json` contains the synthetic inputs and outputs; `provenance.json` hashes that artifact and the Rust sources, and records the commit and dirty-worktree status. `fit.svg` contains the plots.

The teacher and fitted model share the same equations and nearly all parameters. A good result therefore establishes neither parameter identifiability nor biological generalization. For the much harder real-data optimization problem, read the [training-capacity protocol](../docs/LEVEL0-CAPACITY-PROTOCOL.md) and [L-BFGS diagnostic](../docs/CAPACITY-LBFGS.md).